# Notebook — Qualitative figures for the final BAP2

Goal: generate qualitative figures using **the final step4689 checkpoint**, the project's exact pipeline and a consistent inference protocol.

**Model used**
- Base: `SG161222/Realistic_Vision_V4.0_noVAE`
- VAE: `stabilityai/sd-vae-ft-mse`
- Adapter: `h94/IP-Adapter-FaceID`
- BAP2 final: `bap2_s4689.pt`
- Visual prompt: original/simple protocol
- Fixed seed: `42`

This notebook does not recompute metrics. It only generates images and qualitative figures.

In [1]:
# ============================================================
# Cell 1 — Minimal install and IP-Adapter repo
# ============================================================

import os
import sys
import subprocess
import torch

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

pip_install([
    "diffusers",
    "transformers",
    "accelerate",
    "safetensors",
    "huggingface_hub",
])

if not os.path.exists("/content/IP-Adapter"):
    !git clone -q https://github.com/tencent-ailab/IP-Adapter.git /content/IP-Adapter

if "/content/IP-Adapter" not in sys.path:
    sys.path.append("/content/IP-Adapter")

print("✓ Dependencias base listas")
print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

✓ Dependencias base listas
Torch: 2.10.0+cu128
CUDA disponible: True


## Common setup


In [ ]:
# === Common project bootstrap ===
# Bootstrap Colab/Drive + src path
import os, sys

DRIVE = "/content/drive/MyDrive/Notebook4"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

for _path in [DRIVE, "/content", "/content/IP-Adapter"]:
    if _path not in sys.path:
        sys.path.insert(0, _path)


from src.config import DEFAULT_DRIVE
from src.inference_utils import (
    CHILD_NEGATIVE_PROMPT,
    DEFAULT_NEGATIVE_PROMPT,
    collect_bap2_alpha_stats,
    ensure_2d,
    generate_bap2_image,
    get_embedding,
    set_parent_scale_bap2,
)

from src.notebook_setup import setup_full_generation_context

DRIVE = globals().get("DRIVE", DEFAULT_DRIVE)
ctx = setup_full_generation_context(
    DRIVE,
    load_data=True,
    load_arcface=False,
    parent_scale=0.1,
)

paths = ctx.paths
runtime = ctx.runtime
data = ctx.data
pipe = ctx.pipe
ip_model = ctx.ip_model
bap2_modules = ctx.bap2_modules

DEVICE = runtime.device
DTYPE = runtime.dtype
FIW_ROOT = str(paths.fiw_root)
EMBEDDINGS_PATH = str(paths.embeddings_path)
TEST_CSV = str(paths.test_csv)

embeddings = data.embeddings
gender_dict = data.gender_dict
test_ds = data.test_ds
test_loader = data.test_loader

OUT_DIR = f"{DRIVE}/results/figs_bap2"
print('Qualitative figures context ready')
print('BAP2 modules:', len(bap2_modules))


# Helpers cualitativos comunes
import torch
from PIL import Image, ImageOps

CKPT_PATH = str(paths.final_bap2_checkpoint)
SEED = globals().get("SEED", 42)
GUIDANCE = globals().get("GUIDANCE", 4.0)
STEPS = globals().get("STEPS", 30)
PARENT_SCALE = globals().get("PARENT_SCALE", 0.1)

def get_emb(key):
    return get_embedding(embeddings, key)

def key_to_path(key):
    path = os.path.join(FIW_ROOT, str(key))
    assert os.path.exists(path), f"Image does not exist: {path}"
    return path

def prep_img(path_or_pil, size=230):
    if isinstance(path_or_pil, Image.Image):
        img = path_or_pil.convert("RGB")
    else:
        img = Image.open(path_or_pil).convert("RGB")
    return ImageOps.fit(img, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))

@torch.inference_mode()
def generate_bap2_visual_original_prompt(h_c, h_p, h_m, gender, age, seed=42, parent_scale=0.1):
    img, prompt, _alpha_stats = generate_bap2_image(
        pipe=pipe,
        ip_model=ip_model,
        h_c=h_c,
        h_p=h_p,
        h_m=h_m,
        gender=gender,
        age=age,
        device=DEVICE,
        dtype=DTYPE,
        parent_scale=parent_scale,
        seed=seed,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        negative_prompt=(
            "monochrome, lowres, bad anatomy, worst quality, low quality, blurry, "
            "outdoor, brick wall, wooden wall, different background, side view, profile view, "
            "hat, sunglasses, beard, mustache"
        ),
        child_prompt=False,
    )
    return img, prompt

print('Helpers cualitativos listos')


In [35]:
# ============================================================
# Cell 7 — Minimal test with ONE triad
# ============================================================

import os
import matplotlib.pyplot as plt

TEST_TRIAD = 166
TEST_AGES = PROG_AGES

sample = test_ds[TEST_TRIAD]

gender = sample.get("gender", "person")
if gender not in ["man", "woman", "person"]:
    gender = "person"

print("TRIAD_IDX:", TEST_TRIAD)
print("Father:", sample["key_p"])
print("Mother:", sample["key_m"])
print("Child :", sample["key_c"])
print("Gender:", gender)
print("Age real estimada:", sample["age"])

h_c = get_emb(sample["key_c"])
h_p = get_emb(sample["key_p"])
h_m = get_emb(sample["key_m"])

TEST_DIR = f"{OUT_DIR}/single_test_triad_{TEST_TRIAD:04d}"
os.makedirs(TEST_DIR, exist_ok=True)

paths = []

for age in TEST_AGES:
    img, prompt = generate_bap2_visual_original_prompt(
        h_c=h_c,
        h_p=h_p,
        h_m=h_m,
        gender=gender,
        age=age,
        seed=SEED,
        parent_scale=PARENT_SCALE,
    )

    out_path = f"{TEST_DIR}/t{TEST_TRIAD:04d}_a{age:02d}.jpg"
    img.save(out_path, quality=95)

    paths.append((age, out_path, prompt))
    print("✓", age, "|", prompt)

fig, axes = plt.subplots(1, len(TEST_AGES), figsize=(len(TEST_AGES) * 2.2, 2.8), dpi=180)

for ax, (age, path, prompt) in zip(axes, paths):
    img = prep_img(path, size=230)
    ax.imshow(img)
    ax.set_title(str(age), fontsize=9)
    ax.axis("off")

plt.suptitle(f"BAP2 step4689 | triad {TEST_TRIAD} | seed={SEED}", fontsize=11, y=1.03)
plt.tight_layout()

SINGLE_TEST_PNG = f"{TEST_DIR}/grid.png"
fig.savefig(SINGLE_TEST_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("✓ Test saved to:", TEST_DIR)
print("PNG:", SINGLE_TEST_PNG)

TRIAD_IDX: 166
Padre: F0952/MID1/P10025_face0.jpg
Madre: F0952/MID3/P10026_face3.jpg
Hijo : F0952/MID2/P10025_face1.jpg
Gender: man
Age real estimada: 4


  0%|          | 0/30 [00:00<?, ?it/s]

✓ 5 | portrait photo of a man, age 5, realistic, neutral expression, child face, soft youthful features, centered face, plain gray background, soft lighting, natural facial aging


  0%|          | 0/30 [00:00<?, ?it/s]

✓ 15 | portrait photo of a man, age 15, realistic, neutral expression, teenage face, adolescent facial proportions, centered face, plain gray background, soft lighting, natural facial aging


...

✓ 40 | portrait photo of a man, age 40, realistic, neutral expression, adult face, subtle aging signs, centered face, plain gray background, soft lighting, natural facial aging


  0%|          | 0/30 [00:00<?, ?it/s]

✓ 60 | portrait photo of a man, age 60, realistic, neutral expression, older adult face, visible wrinkles, aged skin texture, centered face, plain gray background, soft lighting, natural facial aging


<Figure size 1980x504 with 5 Axes>

✓ Test saved to: /content/drive/MyDrive/Notebook4/results/figs_bap2/test
PNG: /content/drive/MyDrive/Notebook4/results/figs_bap2/test/grid.png


In [10]:
# Generates the SAME age twice with the SAME seed
TEST_TRIAD = 166
AGE_TEST = 20

sample = test_ds[TEST_TRIAD]
gender = sample.get("gender", "person")
if gender not in ["man", "woman", "person"]:
    gender = "person"

h_c = get_emb(sample["key_c"])
h_p = get_emb(sample["key_p"])
h_m = get_emb(sample["key_m"])

img1, _ = generate_bap2_visual_original_prompt(
    h_c, h_p, h_m, gender, AGE_TEST,
    seed=42,
    parent_scale=0.1
)

img2, _ = generate_bap2_visual_original_prompt(
    h_c, h_p, h_m, gender, AGE_TEST,
    seed=42,
    parent_scale=0.1
)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(6,3))
ax[0].imshow(img1); ax[0].set_title("run 1"); ax[0].axis("off")
ax[1].imshow(img2); ax[1].set_title("run 2"); ax[1].axis("off")
plt.show()

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

<Figure size 600x300 with 2 Axes>

In [50]:
# ============================================================
# Cell 8 — Generate the progression for the final triads
# Father | Mother | Real Child | 5 | 10 | 15 | 20 | 30 | 40
# ============================================================

import os
import pandas as pd

FORCE_REGENERATE = False

rows = []

for triad_idx in MANUAL_TRIADS:
    sample = test_ds[int(triad_idx)]

    gender = sample.get("gender", "person")
    if gender not in ["man", "woman", "person"]:
        gender = "person"

    h_c = get_emb(sample["key_c"])
    h_p = get_emb(sample["key_p"])
    h_m = get_emb(sample["key_m"])

    print(f"\nTriada {triad_idx}")
    print("Father:", sample["key_p"])
    print("Mother:", sample["key_m"])
    print("Child :", sample["key_c"])
    print("Gender:", gender, "| estimated real age:", sample["age"])

    for age in PROG_AGES:
        out_path = f"{PROGRESSION_DIR}/t{int(triad_idx):04d}_a{age:02d}.jpg"

        if os.path.exists(out_path) and not FORCE_REGENERATE:
            print("skip exists:", os.path.basename(out_path))
            prompt = f"portrait photo of a {gender}, age {int(age)}, realistic, neutral expression"
        else:
            img, prompt = generate_bap2_visual_original_prompt(
                h_c=h_c,
                h_p=h_p,
                h_m=h_m,
                gender=gender,
                age=age,
                seed=SEED,
                parent_scale=PARENT_SCALE,
            )

            img.save(out_path, quality=95)
            print("✓ generado:", os.path.basename(out_path), "|", prompt)

        rows.append({
            "triad_idx": int(triad_idx),
            "age": int(age),
            "seed": int(SEED),
            "parent_scale": float(PARENT_SCALE),
            "gender": gender,
            "real_age_child": sample["age"],
            "key_p": sample["key_p"],
            "key_m": sample["key_m"],
            "key_c": sample["key_c"],
            "image_path": out_path,
            "prompt": prompt,
        })

progression_df = pd.DataFrame(rows)
PROG_CSV = f"{OUT_DIR}/meta/prog_paths.csv"
progression_df.to_csv(PROG_CSV, index=False)

print("\n✓ CSV saved:", PROG_CSV)
display(progression_df.head(12))


Triada 166
Padre: F0952/MID1/P10025_face0.jpg
Madre: F0952/MID3/P10026_face3.jpg
Hijo : F0952/MID2/P10025_face1.jpg
Gender: man | estimated real age: 4
skip exists: t0166_a05.jpg
skip exists: t0166_a15.jpg
skip exists: t0166_a30.jpg
skip exists: t0166_a45.jpg
skip exists: t0166_a60.jpg

Triada 174
Padre: F0965/MID1/P10163_face2.jpg
Madre: F0965/MID4/P10163_face0.jpg
Hijo : F0965/MID5/P10163_face1.jpg
Gender: woman | estimated real age: 5
skip exists: t0174_a05.jpg
skip exists: t0174_a15.jpg
skip exists: t0174_a30.jpg
skip exists: t0174_a45.jpg
skip exists: t0174_a60.jpg

Triada 71
Padre: F0427/MID5/P04490_face2.jpg
Madre: F0427/MID1/P04490_face0.jpg
Hijo : F0427/MID3/P04490_face1.jpg
Gender: woman | estimated real age: 11
skip exists: t0071_a05.jpg
skip exists: t0071_a15.jpg
skip exists: t0071_a30.jpg


  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: t0071_a45.jpg | portrait photo of a woman, age 45, realistic, neutral expression, adult face, subtle aging signs, centered face, plain gray background, soft lighting, natural facial aging
skip exists: t0071_a60.jpg

Triada 103
Padre: F0588/MID5/P06193_face3.jpg
Madre: F0588/MID1/P06190_face0.jpg
Hijo : F0588/MID4/P06192_face0.jpg
Gender: man | estimated real age: 1
skip exists: t0103_a05.jpg
skip exists: t0103_a15.jpg
skip exists: t0103_a30.jpg
skip exists: t0103_a45.jpg
skip exists: t0103_a60.jpg

Triada 202
Padre: F1005/MID1/P13041_face0.jpg
Madre: F1005/MID2/P13041_face1.jpg
Hijo : F1005/MID4/P13041_face3.jpg
Gender: man | estimated real age: 8
skip exists: t0202_a05.jpg
skip exists: t0202_a15.jpg
skip exists: t0202_a30.jpg
skip exists: t0202_a45.jpg
skip exists: t0202_a60.jpg

Triada 173
Padre: F0965/MID1/P10163_face2.jpg
Madre: F0965/MID2/P10164_face3.jpg
Hijo : F0965/MID3/P10166_face1.jpg
Gender: man | estimated real age: 4
skip exists: t0173_a05.jpg
skip exists: t017

,triad_idx,age,seed,parent_scale,gender,real_age_child,key_p,key_m,key_c,image_path,prompt
0,166,5,42,0.1,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a man, age 5, realistic, neu..."
1,166,15,42,0.1,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a man, age 15, realistic, ne..."
2,166,30,42,0.1,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a man, age 30, realistic, ne..."
3,166,45,42,0.1,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a man, age 45, realistic, ne..."
4,166,60,42,0.1,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a man, age 60, realistic, ne..."
5,174,5,42,0.1,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a woman, age 5, realistic, n..."
6,174,15,42,0.1,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a woman, age 15, realistic, ..."
7,174,30,42,0.1,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a woman, age 30, realistic, ..."
8,174,45,42,0.1,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a woman, age 45, realistic, ..."
9,174,60,42,0.1,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,/content/drive/MyDrive/Notebook4/results/final...,"portrait photo of a woman, age 60, realistic, ..."


In [51]:
# ============================================================
# Cell 9 — Build the final progression figure
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROG_CSV = f"{OUT_DIR}/meta/prog_paths.csv"
assert os.path.exists(PROG_CSV), PROG_CSV

progression_df = pd.read_csv(PROG_CSV)

base_rows = (
    progression_df[["triad_idx", "gender", "real_age_child", "key_p", "key_m", "key_c"]]
    .drop_duplicates()
    .copy()
)

base_rows["manual_order"] = base_rows["triad_idx"].apply(lambda x: MANUAL_TRIADS.index(int(x)))
base_rows = base_rows.sort_values("manual_order").reset_index(drop=True)

col_titles = ["Father", "Mother", "Real Child"] + [str(a) for a in PROG_AGES]

n_rows = len(base_rows)
n_cols = len(col_titles)

fig_w = n_cols * 1.75
fig_h = n_rows * 1.85 + 0.7

fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), dpi=300)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, row in base_rows.iterrows():
    triad_idx = int(row["triad_idx"])

    fixed_paths = [
        key_to_path(row["key_p"]),
        key_to_path(row["key_m"]),
        key_to_path(row["key_c"]),
    ]

    gen_paths = []
    for age in PROG_AGES:
        match = progression_df[
            (progression_df["triad_idx"] == triad_idx) &
            (progression_df["age"] == age)
        ]
        assert len(match) > 0, f"Missing image triad={triad_idx}, age={age}"
        gen_paths.append(match.iloc[0]["image_path"])

    all_paths = fixed_paths + gen_paths

    for c, path in enumerate(all_paths):
        ax = axes[r, c]
        img = prep_img(path, size=230)
        ax.imshow(img)
        ax.axis("off")

        if r == 0:
            ax.set_title(col_titles[c], fontsize=9, pad=5)

    label = f"T{triad_idx}\nreal={float(row['real_age_child']):.1f}"
    axes[r, 0].text(
        -0.23, 0.5,
        label,
        transform=axes[r, 0].transAxes,
        rotation=90,
        va="center",
        ha="center",
        fontsize=7
    )

plt.subplots_adjust(
    left=0.055,
    right=0.995,
    top=0.955,
    bottom=0.02,
    wspace=0.03,
    hspace=0.055
)

fig.suptitle("Age progression conditioned by parental features — BAP2 step4689", fontsize=12, y=0.99)

FIG_PNG = f"{OUT_DIR}/figure_progression_bap2_seed42.png"

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight", facecolor="white")

plt.show()

print("✓ Figura PNG:", FIG_PNG)


<Figure size 4200x3540 with 48 Axes>

✓ Figura PNG: /content/drive/MyDrive/Notebook4/results/figs_bap2/figure_progression_bap2_seed42.png


## Methodological note

This notebook uses the **final step4689 checkpoint**, so the figures correspond to the reported final model.  
The visual protocol uses a fixed seed and a simple prompt to make the progression more consistent without changing the model.

Do not use these images to recompute quantitative tables unless all baselines are regenerated under exactly the same protocol.